# Fraud Detection Pipeline (`orders.is_fraud`) using CRISP-DM

## CRISP-DM Roadmap
1. **Business Understanding** *(this notebook section now)*
2. **Data Understanding** *(this notebook section now)*
3. **Data Preparation** *(next: feature engineering + train/validation split strategy)*
4. **Modeling** *(next: baseline + tuned models)*
5. **Evaluation** *(next: thresholding + business-cost analysis)*
6. **Deployment** *(next: batch/real-time scoring + monitoring)*

This notebook currently implements the first two phases in depth and prepares outputs needed for phases 3-6.

## Business Understanding

### Fraud-detection problem definition
We need to predict whether an order in the `orders` table is fraudulent (`is_fraud = 1`) before fulfillment decisions are finalized. This is a **binary classification** problem with class imbalance.

### Business objective
Reduce fraud losses while minimizing friction for legitimate customers.

### Decision context
Model outputs will be used to support operational actions such as:
- auto-approve low-risk orders,
- route medium-risk orders to manual review,
- auto-hold/cancel very high-risk orders.

### Success criteria
**Business KPIs**
- Lower net fraud loss rate (chargebacks + write-offs).
- Keep false-positive rate low enough to avoid unnecessary customer friction.
- Maintain acceptable manual-review volume.

**ML metrics**
- Prioritize **Recall** and **PR-AUC** for fraud class.
- Track precision at an operating threshold (e.g., top-k risk queue).
- Track ROC-AUC as a secondary separability metric.

### Constraints and assumptions
- Data source is `shop.db`; target leakage must be avoided.
- Features available at scoring time should be preferred.
- Since fraud prevalence is low, threshold selection must be business-cost aware.

## Data Understanding

In [1]:
# 2) Data Understanding - setup and data loading
import sqlite3
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

DB_PATH = "shop.db"

conn = sqlite3.connect(DB_PATH)

# Load core tables needed for understanding fraud dynamics
orders_df = pd.read_sql_query("SELECT * FROM orders", conn)
customers_df = pd.read_sql_query("SELECT * FROM customers", conn)
shipments_df = pd.read_sql_query("SELECT * FROM shipments", conn)
order_items_df = pd.read_sql_query("SELECT * FROM order_items", conn)

print("orders shape:", orders_df.shape)
print("customers shape:", customers_df.shape)
print("shipments shape:", shipments_df.shape)
print("order_items shape:", order_items_df.shape)
orders_df.head()

orders shape: (5000, 17)
customers shape: (250, 12)
shipments shape: (5000, 9)
order_items shape: (15022, 6)


,order_id,customer_id,order_datetime,billing_zip,shipping_zip,shipping_state,payment_method,device_type,ip_country,promo_used,promo_code,order_subtotal,shipping_fee,tax_amount,order_total,risk_score,is_fraud
0,1,1,2025-11-29 00:51:07,28289,28289,CO,card,mobile,US,0,None,662.9500,15.4400,46.3000,724.6900,38.3000,0
1,2,1,2025-09-01 10:25:59,28289,13888,NY,card,desktop,US,1,SAVE10,862.9200,14.7400,66.6100,944.2700,94.9000,0
2,3,1,2025-12-15 07:24:41,28289,28289,CO,card,mobile,US,0,None,796.0900,14.0400,40.7200,850.8500,53.8000,1
3,4,1,2025-11-06 18:21:19,28289,28289,CO,bank,mobile,US,1,WELCOME,137.6000,6.9900,11.8800,156.4700,4.2000,0
4,5,1,2025-11-30 05:34:15,28289,28289,CO,card,mobile,CA,0,None,17.0700,6.9900,1.4000,25.4600,4.9000,0


In [2]:
# Target distribution and data quality checks
n_orders = len(orders_df)
fraud_counts = orders_df["is_fraud"].value_counts().sort_index()
fraud_rate = orders_df["is_fraud"].mean()

print("Total orders:", n_orders)
print("\nFraud class counts:")
print(fraud_counts)
print(f"\nFraud prevalence: {fraud_rate:.2%}")

missing_pct = (orders_df.isna().mean() * 100).sort_values(ascending=False)
print("\nMissingness (% of rows):")
print(missing_pct[missing_pct > 0])

print("\nDtypes:")
print(orders_df.dtypes)

Total orders: 5000

Fraud class counts:
is_fraud
0    4682
1     318
Name: count, dtype: int64

Fraud prevalence: 6.36%

Missingness (% of rows):
promo_code   74.7800
dtype: float64

Dtypes:
order_id            int64
customer_id         int64
order_datetime     object
billing_zip        object
shipping_zip       object
shipping_state     object
payment_method     object
device_type        object
ip_country         object
promo_used          int64
promo_code         object
order_subtotal    float64
shipping_fee      float64
tax_amount        float64
order_total       float64
risk_score        float64
is_fraud            int64
dtype: object


In [3]:
# Feature-level exploration inside orders
categorical_cols = [
    "payment_method", "device_type", "ip_country", "shipping_state", "promo_used"
]

for col in categorical_cols:
    rates = (
        orders_df.groupby(col)["is_fraud"]
        .agg(["mean", "count"])
        .rename(columns={"mean": "fraud_rate", "count": "n_orders"})
        .sort_values("fraud_rate", ascending=False)
    )
    print(f"\nFraud rate by {col}:")
    print(rates)

numeric_cols = ["order_subtotal", "shipping_fee", "tax_amount", "order_total", "risk_score"]
print("\nNumeric feature means by class:")
print(orders_df.groupby("is_fraud")[numeric_cols].mean())

print("\nNumeric feature medians by class:")
print(orders_df.groupby("is_fraud")[numeric_cols].median())


Fraud rate by payment_method:
                fraud_rate  n_orders
payment_method                      
crypto              0.1031        97
card                0.0675      3128
bank                0.0593       725
paypal              0.0514      1050

Fraud rate by device_type:
             fraud_rate  n_orders
device_type                      
mobile           0.0680      2734
tablet           0.0659       364
desktop          0.0568      1902

Fraud rate by ip_country:
            fraud_rate  n_orders
ip_country                      
IN              0.0947        95
GB              0.0865       104
BR              0.0732        41
CA              0.0688       218
NG              0.0652        46
US              0.0621      4496

Fraud rate by shipping_state:
                fraud_rate  n_orders
shipping_state                      
CO                  0.0917      1702
TX                  0.0829       350
OR                  0.0755       106
VA                  0.0638        47
WA   

In [4]:
# Relationship discovery across tables
item_agg_df = pd.read_sql_query(
    """
    SELECT
        oi.order_id,
        COUNT(*) AS item_count,
        SUM(oi.quantity) AS total_qty,
        AVG(oi.unit_price) AS avg_unit_price,
        SUM(oi.line_total) AS line_total_sum
    FROM order_items oi
    GROUP BY oi.order_id
    """,
    conn,
)

analysis_df = (
    orders_df
    .merge(item_agg_df, on="order_id", how="left")
    .merge(
        shipments_df[[
            "order_id", "carrier", "shipping_method", "distance_band",
            "promised_days", "actual_days", "late_delivery"
        ]],
        on="order_id",
        how="left",
    )
)

corr_cols = [
    "is_fraud", "risk_score", "order_total", "item_count", "total_qty",
    "avg_unit_price", "promised_days", "actual_days", "late_delivery"
]

print("Correlations with is_fraud:")
print(
    analysis_df[corr_cols]
    .corr(numeric_only=True)["is_fraud"]
    .sort_values(ascending=False)
)

for col in ["shipping_method", "carrier", "distance_band", "late_delivery"]:
    rates = (
        analysis_df.groupby(col)["is_fraud"]
        .agg(["mean", "count"])
        .rename(columns={"mean": "fraud_rate", "count": "n_orders"})
        .sort_values("fraud_rate", ascending=False)
    )
    print(f"\nFraud rate by {col}:")
    print(rates)

Correlations with is_fraud:
is_fraud          1.0000
actual_days       0.3205
risk_score        0.2701
late_delivery     0.2129
order_total       0.2062
total_qty         0.1412
item_count        0.1229
avg_unit_price    0.1026
promised_days    -0.0010
Name: is_fraud, dtype: float64

Fraud rate by shipping_method:
                 fraud_rate  n_orders
shipping_method                      
overnight            0.0792       303
standard             0.0638      3618
expedited            0.0584      1079

Fraud rate by carrier:
         fraud_rate  n_orders
carrier                      
USPS         0.0672       997
FedEx        0.0639      1705
UPS          0.0618      2298

Fraud rate by distance_band:
               fraud_rate  n_orders
distance_band                      
national           0.0727       990
local              0.0688      1716
regional           0.0558      2294

Fraud rate by late_delivery:
               fraud_rate  n_orders
late_delivery                      
1       

## Key Data Understanding Findings (from current dataset)
- Fraud prevalence is about **6.36%** (class imbalance exists).
- `risk_score` and `order_total` are materially higher for fraudulent orders.
- Fraud rates vary by `payment_method`, `ip_country`, and `shipping_method`.
- Aggregated basket features (`item_count`, `total_qty`, `avg_unit_price`) have signal.
- Shipment outcomes such as `late_delivery` show a strong relationship with fraud in this dataset.

## 3) Data Preparation

Goals for this phase:
1. **Merge** all relevant tables into a single analytical dataset at the order level.
2. **Engineer features** from raw columns (temporal, basket-level, address mismatch, customer demographics).
3. **Identify and exclude** leaky or non-predictive columns.
4. **Build a reproducible `sklearn` preprocessing pipeline** (imputation, encoding, scaling).
5. **Split** into train / test sets using a time-aware strategy on `order_datetime`.

In [ ]:
# 3a) Merge tables into a single order-level analytical dataset

item_agg = pd.read_sql_query(
    """
    SELECT
        order_id,
        COUNT(*)          AS item_count,
        SUM(quantity)     AS total_qty,
        AVG(unit_price)   AS avg_unit_price,
        MAX(unit_price)   AS max_unit_price,
        MIN(unit_price)   AS min_unit_price
    FROM order_items
    GROUP BY order_id
    """,
    conn,
)

ship_cols = [
    "order_id", "carrier", "shipping_method", "distance_band",
    "promised_days", "actual_days", "late_delivery",
]

cust_cols = [
    "customer_id", "gender", "birthdate", "created_at",
    "customer_segment", "loyalty_tier",
]

df = (
    orders_df
    .merge(item_agg, on="order_id", how="left")
    .merge(shipments_df[ship_cols], on="order_id", how="left")
    .merge(customers_df[cust_cols], on="customer_id", how="left")
)

print("Merged dataset shape:", df.shape)
df.head(3)

In [ ]:
# 3b) Feature engineering

df["order_datetime"] = pd.to_datetime(df["order_datetime"])

# Temporal features
df["order_hour"] = df["order_datetime"].dt.hour
df["order_dow"] = df["order_datetime"].dt.dayofweek          # 0=Mon … 6=Sun
df["order_is_weekend"] = (df["order_dow"] >= 5).astype(int)

# Address mismatch: billing zip != shipping zip
df["zip_mismatch"] = (df["billing_zip"] != df["shipping_zip"]).astype(int)

# Customer account age at order time (days)
df["customer_created_at"] = pd.to_datetime(df["created_at"])
df["account_age_days"] = (
    (df["order_datetime"] - df["customer_created_at"]).dt.total_seconds() / 86400
).clip(lower=0)

# Customer age at order time (years) from birthdate
df["birthdate"] = pd.to_datetime(df["birthdate"], errors="coerce")
df["customer_age_years"] = (
    (df["order_datetime"] - df["birthdate"]).dt.total_seconds() / (86400 * 365.25)
)

# Price spread in basket
df["price_range"] = df["max_unit_price"] - df["min_unit_price"]

# Delivery gap (promised vs actual) — available post-shipment but useful as a feature
df["delivery_gap"] = df["actual_days"] - df["promised_days"]

print("Engineered features added. New shape:", df.shape)
df[["order_hour", "order_dow", "order_is_weekend", "zip_mismatch",
    "account_age_days", "customer_age_years", "price_range", "delivery_gap"]].describe()

In [ ]:
# 3c) Define prediction-time-safe feature set and drop leaky / ID columns
#
# Leakage notes:
#   - `actual_days`, `late_delivery`, `delivery_gap` are post-fulfillment
#     outcomes that correlate strongly with fraud but would NOT be available
#     when scoring a new order in real time.  We exclude them.
#   - `risk_score` appears to be a pre-existing scoring signal available
#     at order time, so we keep it.
#   - IDs, free-text codes, and raw datetime are not model features.

DROP_COLS = [
    "order_id", "customer_id",           # identifiers
    "order_datetime",                     # used for split, not as feature
    "billing_zip", "shipping_zip",        # replaced by zip_mismatch
    "promo_code",                         # high cardinality + 75 % missing
    "created_at", "customer_created_at",  # used to derive account_age_days
    "birthdate",                          # used to derive customer_age_years
    "actual_days", "late_delivery",       # post-fulfillment leakage
    "delivery_gap",                       # derived from actual_days (leaky)
    "is_fraud",                           # target — separated below
]

TARGET = "is_fraud"
y = df[TARGET].copy()
X = df.drop(columns=DROP_COLS, errors="ignore")

print("Feature matrix shape:", X.shape)
print("Target distribution:\n", y.value_counts())
print("\nFeature columns:")
print(list(X.columns))

In [ ]:
# 3d) Build sklearn preprocessing pipeline (ColumnTransformer)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "string"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("encoder", OneHotEncoder(handle_unknown="infrequent_if_exist",
                              min_frequency=20,
                              sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])

In [ ]:
# 3e) Time-aware train / test split
#
# We sort orders chronologically and use the earliest 80% for training,
# the most recent 20% for testing.  This respects the temporal ordering
# that would exist in a real deployment (no future leakage into training).

df_sorted = df.sort_values("order_datetime").reset_index(drop=True)
split_idx = int(len(df_sorted) * 0.80)

train_idx = df_sorted.index[:split_idx]
test_idx = df_sorted.index[split_idx:]

X_train, X_test = X.loc[train_idx], X.loc[test_idx]
y_train, y_test = y.loc[train_idx], y.loc[test_idx]

print(f"Train: {len(X_train)} orders  |  Fraud rate: {y_train.mean():.2%}")
print(f"Test : {len(X_test)} orders  |  Fraud rate: {y_test.mean():.2%}")
print(f"\nTrain date range: {df_sorted.loc[train_idx, 'order_datetime'].min()} → "
      f"{df_sorted.loc[train_idx, 'order_datetime'].max()}")
print(f"Test  date range: {df_sorted.loc[test_idx, 'order_datetime'].min()} → "
      f"{df_sorted.loc[test_idx, 'order_datetime'].max()}")

In [ ]:
# 3f) Sanity-check: fit preprocessor on training data, inspect output shape
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

ohe_feature_names = (
    preprocessor.named_transformers_["cat"]
    .named_steps["encoder"]
    .get_feature_names_out(categorical_features)
    .tolist()
)
all_feature_names = numeric_features + ohe_feature_names

print(f"Processed train shape: {X_train_processed.shape}")
print(f"Processed test  shape: {X_test_processed.shape}")
print(f"\nTotal features after encoding: {len(all_feature_names)}")
print("Feature names:", all_feature_names)

### Data Preparation Summary

| Step | Detail |
|------|--------|
| **Merge** | `orders` + aggregated `order_items` + `shipments` + `customers` into one order-level row |
| **Engineered features** | `order_hour`, `order_dow`, `order_is_weekend`, `zip_mismatch`, `account_age_days`, `customer_age_years`, `price_range` |
| **Leakage exclusions** | `actual_days`, `late_delivery`, `delivery_gap` (post-fulfillment); IDs; raw dates; `promo_code` (75 % missing + high cardinality) |
| **Numeric pipeline** | Median imputation → StandardScaler |
| **Categorical pipeline** | Constant imputation ("missing") → OneHotEncoder (rare categories grouped via `min_frequency=20`) |
| **Split strategy** | Chronological 80/20 on `order_datetime` to prevent temporal leakage |

The `preprocessor` ColumnTransformer and `X_train` / `X_test` / `y_train` / `y_test` objects are ready for the Modeling phase.

## Modeling
- Baseline: Logistic Regression.
- Candidate models: Random Forest, Gradient Boosting / XGBoost-like model.
- Handle imbalance using class weights and/or resampling.

## Evaluation
- Optimize threshold using business cost matrix (false negatives vs false positives).
- Report PR-AUC, Recall, Precision, F1, ROC-AUC, and confusion matrix.
- Validate model calibration and stability over time windows.

## Deployment
- Save model + preprocessing artifacts as one reproducible pipeline.
- Define scoring interface (batch or API).
- Add monitoring: data drift, concept drift, fraud-rate shift, and threshold re-tuning cadence.